# YSSY LightGBM Wind Forecasting Workflow

This notebook explains and runs the `YSSY_LightGBM.py` workflow step by step.

The purpose of this workflow is to train a **LightGBM multi-output regression model** to forecast future wind components at **YSSY**.  
The model predicts future `u` and `v` wind components for multiple forecast horizons, using prepared weather features and lagged wind information.

The workflow includes:

1. Loading training, validation, and test datasets  
2. Identifying target columns and input feature columns  
3. Preparing `X` and `y` datasets  
4. Loading or tuning LightGBM hyperparameters  
5. Training a multi-output LightGBM model  
6. Plotting predicted future winds against actual future winds  
7. Evaluating the model using MAE and MSE  
8. Saving the trained model and output plots


## 1. Import Required Libraries

This section imports the Python libraries used in the workflow.

Key libraries include:

- `pandas` and `numpy` for data handling
- `lightgbm` for the regression model
- `MultiOutputRegressor` for predicting multiple future outputs
- `optuna` for hyperparameter tuning
- `matplotlib` for plotting forecast results
- `joblib` and `json` for saving model artefacts and parameters


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import optuna
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import joblib
import json
from datetime import timedelta
from pathlib import Path
import random


## 2. Configuration

This section defines the file paths and output locations.

The script expects the prepared datasets to be stored in the `data24x0.1` folder:

- `training_dataset.txt`
- `validation_dataset.txt`
- `test_dataset.txt`

The outputs are saved in the `output` folder, including:

- best parameter files
- trained model file
- forecast plots


In [ ]:
TIMESTAMP_COL = "timestamp_t"

DATA_DIR = Path("data24x0.1")
TRAIN_FILE = DATA_DIR / "training_dataset.txt"
VAL_FILE = DATA_DIR / "validation_dataset.txt"
TEST_FILE = DATA_DIR / "test_dataset.txt"

# This raw file is not used in the current workflow because lagged YSSY features are already included in the main datasets.
YSSY_RAW_FILE = DATA_DIR / "YSSY.txt"

PARAMS_FILE_PATH = Path("output") / "best_params_final_31May.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

BEST_PARAMS_INITIAL_FILE = OUTPUT_DIR / "best_params_initial.json"
BEST_PARAMS_FINAL_FILE = OUTPUT_DIR / "best_params_final.json"
FINAL_MODEL_FILE = OUTPUT_DIR / "final_yssy_wind_model.joblib"

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print("Training file:", TRAIN_FILE)
print("Validation file:", VAL_FILE)
print("Test file:", TEST_FILE)
print("Output directory:", OUTPUT_DIR.resolve())


## 3. Identify Target and Feature Columns

The function `get_column_names()` automatically identifies important column groups.

The target columns are the future YSSY wind component columns, for example:

- `YSSY_u_forecast_t+1`
- `YSSY_v_forecast_t+1`
- `YSSY_u_forecast_t+2`
- `YSSY_v_forecast_t+2`

If the data interval is 30 minutes, then `t+1` means 30 minutes ahead, and `t+12` means 6 hours ahead.

The function also identifies YSSY lagged input wind columns, such as:

- `YSSY_u_component_lag0`
- `YSSY_u_component_lag1`
- `YSSY_v_component_lag0`
- `YSSY_v_component_lag1`

These are used later for plotting the past wind input sequence.


In [ ]:
def get_column_names(df_columns):
    """Identifies target, drop, and YSSY lagged wind columns."""

    target_cols = [
        col for col in df_columns
        if col.startswith("YSSY_u_forecast_t+") or col.startswith("YSSY_v_forecast_t+")
    ]

    target_cols = sorted(
        target_cols,
        key=lambda x: (int(x.split('+')[-1]), x.split('_')[1])
    )

    cols_to_drop_explicit = [
        "BELL_mslp_deriv_t_vs_t3",
        "BELL_mslp_deriv_t3_vs_t6",
        "BELL_mslp_deriv_t_vs_t6",
        "MTB_mslp_deriv_t_vs_t3",
        "MTB_mslp_deriv_t3_vs_t6",
        "MTB_mslp_deriv_t_vs_t6"
    ]

    cols_to_drop_from_x = [
        col for col in cols_to_drop_explicit
        if col in df_columns
    ]

    yssy_u_lag_cols = sorted(
        [c for c in df_columns if c.startswith("YSSY_u_component_lag")],
        key=lambda x: int(x.split('lag')[-1])
    )

    yssy_v_lag_cols = sorted(
        [c for c in df_columns if c.startswith("YSSY_v_component_lag")],
        key=lambda x: int(x.split('lag')[-1])
    )

    return target_cols, cols_to_drop_from_x, yssy_u_lag_cols, yssy_v_lag_cols


## 4. Load and Prepare Data

The function `load_and_prepare_data()` reads a dataset and separates it into:

- `df`: the full original dataframe
- `X`: model input features
- `y`: model target outputs
- `timestamps`: timestamps for each row

The target columns and timestamp column are removed from `X`, because the model should not directly use future target values as input.


In [ ]:
def load_and_prepare_data(file_path, target_cols, cols_to_drop_from_x, ts_col=TIMESTAMP_COL):
    """Loads data, separates features/targets, drops specified columns."""

    print(f"Loading data from: {file_path}")
    df = pd.read_csv(file_path)
    df[ts_col] = pd.to_datetime(df[ts_col])

    y = df[target_cols].copy()
    timestamps = df[ts_col].copy()

    cols_to_remove_for_x = target_cols + [ts_col] + cols_to_drop_from_x
    X = df.drop(columns=[col for col in cols_to_remove_for_x if col in df.columns])

    print(f"Loaded {file_path}: X shape {X.shape}, y shape {y.shape}")
    return df, X, y, timestamps


def load_hyperparameters(file_path):
    """Loads hyperparameters from a JSON file."""

    if not file_path.exists():
        print(f"Error: Hyperparameter file not found at {file_path}")
        return None

    try:
        with open(file_path, "r") as f:
            params = json.load(f)
        print(f"Successfully loaded hyperparameters from {file_path}")
        return params

    except Exception as e:
        print(f"Error loading hyperparameters from {file_path}: {e}")
        return None


## 5. Optuna Objective Function

This function is intended to tune LightGBM hyperparameters using Optuna.

However, the current version contains an important issue:

The function first defines a search space using `trial.suggest_*()`, but then immediately overwrites it with a fixed parameter dictionary.  
This means Optuna does **not actually test different parameter combinations** in the current version.

This is acceptable for a quick experiment, but it should be fixed before using this as a final tuning pipeline.


In [ ]:
def optuna_objective(trial, X_train, y_train, X_val, y_val):
    """Optuna objective function for hyperparameter tuning."""

    objective_type = trial.suggest_categorical(
        "objective",
        ["regression_l1", "regression_l2"]
    )
    metric_type = "l1" if objective_type == "regression_l1" else "l2"

    # Intended Optuna search space.
    params = {
        "objective": objective_type,
        "metric": metric_type,
        "n_estimators": trial.suggest_int("n_estimators", 500, 1500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 1000, step=50),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 500),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    # Current fixed parameters.
    # Note: this overwrites the Optuna-generated parameters above.
    # Therefore, Optuna is not really tuning in the current version.
    params = {
        "objective": "regression_l2",
        "metric": "l2",
        "n_estimators": 800,
        "learning_rate": 0.02130325383067257,
        "num_leaves": 200,
        "max_depth": 17,
        "min_child_samples": 45,
        "subsample": 0.8767980400385894,
        "colsample_bytree": 0.6947769061520032,
        "reg_alpha": 1.959826832632918,
        "reg_lambda": 0.12507898430314643,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    print(f"Trial {trial.number} Parameters: {params}")

    base_lgbm = lgb.LGBMRegressor(**params)
    multi_output_model = MultiOutputRegressor(base_lgbm)

    multi_output_model.fit(X_train, y_train)
    preds = multi_output_model.predict(X_val)

    score = mean_squared_error(y_val, preds)
    return score


## 6. Model Training Function

The function `train_final_lgbm_model()` trains a `MultiOutputRegressor` using LightGBM as the base model.

This is needed because the target contains multiple future outputs, such as future `u` and `v` components for multiple forecast horizons.

Conceptually, `MultiOutputRegressor` trains one LightGBM model for each target column.


In [ ]:
def train_final_lgbm_model(X_train, y_train, params):
    """Trains a MultiOutputRegressor with LGBM using given parameters."""

    if params is None:
        raise ValueError("No hyperparameters were provided. Please check the parameter file path.")

    lgbm_params = params.copy()

    if "metric" not in lgb.LGBMRegressor().get_params().keys():
        lgbm_params.pop("metric", None)

    base_lgbm = lgb.LGBMRegressor(
        **lgbm_params,
        random_state=42,
        n_jobs=-1
    )

    model = MultiOutputRegressor(base_lgbm)
    model.fit(X_train, y_train)

    return model


## 7. Model Evaluation Function

The function `evaluate_model_performance()` evaluates the trained model on the test set.

It reports:

- overall MAE
- overall MSE
- MAE and MSE for each forecast target

This is useful for checking whether short-term forecasts are more accurate than longer-term forecasts.


In [ ]:
def evaluate_model_performance(model, X_test, y_test, target_names):
    """Evaluates the model and prints metrics."""

    print("\nEvaluating model performance...")
    preds = model.predict(X_test)

    overall_mae = mean_absolute_error(y_test, preds)
    overall_mse = mean_squared_error(y_test, preds)

    print(f"Overall MAE: {overall_mae:.4f}")
    print(f"Overall MSE: {overall_mse:.4f}")

    print("\nPer-target metrics:")
    for i, target_name in enumerate(target_names):
        mae = mean_absolute_error(y_test.iloc[:, i], preds[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], preds[:, i])
        print(f"  {target_name}: MAE={mae:.4f}, MSE={mse:.4f}")

    return overall_mae, overall_mse


## 8. Plotting Functions

The plotting functions compare:

- past YSSY wind components used as model input
- actual future YSSY wind components
- predicted future YSSY wind components

Each plot contains two subplots:

1. U-component forecast  
2. V-component forecast  

The plots are saved into the `output/plots` folder.


In [ ]:
def plot_single_forecast(timestamp_anchor, model, full_df_row, target_names,
                         yssy_u_lag_cols, yssy_v_lag_cols, x_cols_for_model, plot_idx=""):
    """Plots past, actual future, and forecasted future winds for a sample."""

    past_u_winds = full_df_row[yssy_u_lag_cols].values[::-1]
    past_v_winds = full_df_row[yssy_v_lag_cols].values[::-1]

    past_times = [
        timestamp_anchor - timedelta(minutes=30 * i)
        for i in range(11, -1, -1)
    ]

    future_times = [
        timestamp_anchor + timedelta(minutes=30 * i)
        for i in range(1, 13)
    ]

    actual_future_winds_flat = full_df_row[target_names].values
    actual_future_u = actual_future_winds_flat[0::2]
    actual_future_v = actual_future_winds_flat[1::2]

    X_sample = pd.DataFrame(
        [full_df_row[x_cols_for_model]],
        columns=x_cols_for_model
    )

    pred_future_winds_flat = model.predict(X_sample)[0]
    pred_future_u = pred_future_winds_flat[0::2]
    pred_future_v = pred_future_winds_flat[1::2]

    fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    fig.suptitle(
        f"YSSY Wind Forecast anchored at {timestamp_anchor.strftime('%Y-%m-%d %H:%M')}",
        fontsize=16
    )

    axs[0].plot(past_times, past_u_winds, "o-", color="gray", label="Past U (Input)")
    axs[0].plot(future_times, actual_future_u, "s-", color="blue", label="Actual Future U")
    axs[0].plot(future_times, pred_future_u, "x--", color="red", label="Forecasted U")
    axs[0].set_ylabel("U-component")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].plot(past_times, past_v_winds, "o-", color="gray", label="Past V (Input)")
    axs[1].plot(future_times, actual_future_v, "s-", color="green", label="Actual Future V")
    axs[1].plot(future_times, pred_future_v, "x--", color="orange", label="Forecasted V")
    axs[1].set_ylabel("V-component")
    axs[1].legend()
    axs[1].grid(True)

    axs[1].set_xlabel("Time")
    axs[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))

    plt.xticks(rotation=45)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    plot_filename = PLOTS_DIR / f"forecast_plot_{timestamp_anchor.strftime('%Y%m%d_%H%M')}{plot_idx}.png"
    plt.savefig(plot_filename)

    print(f"Saved plot: {plot_filename}")
    plt.close(fig)


def plot_random_samples(model, test_df_full, target_names, yssy_u_lag_cols, yssy_v_lag_cols,
                        x_cols_for_model, num_samples=5):
    """Generates plots for a number of random samples from the test set."""

    print(f"\nGenerating {num_samples} random sample plots...")

    if len(test_df_full) < num_samples:
        print(f"Warning: Requested {num_samples} samples, but test set only has {len(test_df_full)}.")
        num_samples = len(test_df_full)

    random_indices = random.sample(range(len(test_df_full)), num_samples)

    for i, idx in enumerate(random_indices):
        sample_row = test_df_full.iloc[idx]
        anchor_ts = sample_row[TIMESTAMP_COL]

        plot_single_forecast(
            anchor_ts,
            model,
            sample_row,
            target_names,
            yssy_u_lag_cols,
            yssy_v_lag_cols,
            x_cols_for_model,
            plot_idx=f"_rand{i+1}"
        )


## 9. Utility Functions

These helper functions save and load:

- trained model files
- JSON parameter files

The trained LightGBM multi-output model is saved using `joblib`.


In [ ]:
def save_model_artifact(model_obj, path):
    joblib.dump(model_obj, path)
    print(f"Model saved to {path}")


def load_model_artifact(path):
    model_obj = joblib.load(path)
    print(f"Model loaded from {path}")
    return model_obj


def save_json(data_dict, path):
    with open(path, "w") as f:
        json.dump(data_dict, f, indent=4)
    print(f"Parameters saved to {path}")


def load_json(path):
    with open(path, "r") as f:
        data_dict = json.load(f)
    print(f"Parameters loaded from {path}")
    return data_dict


## 10. Main Workflow

This section runs the full workflow.

Steps:

1. Read the training file header to identify target and feature columns  
2. Load training, validation, and test datasets  
3. Create a smaller training sample for quick testing  
4. Run a small Optuna tuning process  
5. Load saved hyperparameters  
6. Train a temporary LightGBM multi-output model  
7. Generate forecast plots  
8. Save the model  
9. Evaluate the model on the test set  

Note: In the current version, this workflow is still closer to a **quick experiment version** than a final training pipeline.


In [ ]:
# 1. Identify column names.
print("Identifying column names...")

sample_df_cols = pd.read_csv(TRAIN_FILE, nrows=0).columns

TARGET_COL_NAMES, COLS_TO_DROP_FROM_X, YSSY_U_LAG_COLS, YSSY_V_LAG_COLS = get_column_names(sample_df_cols)

print(f"Target columns ({len(TARGET_COL_NAMES)}): {TARGET_COL_NAMES[:4]}...")
print(f"Columns to drop from X ({len(COLS_TO_DROP_FROM_X)}): {COLS_TO_DROP_FROM_X}")
print(f"YSSY U lagged input cols ({len(YSSY_U_LAG_COLS)}): {YSSY_U_LAG_COLS[:4]}...")
print(f"YSSY V lagged input cols ({len(YSSY_V_LAG_COLS)}): {YSSY_V_LAG_COLS[:4]}...")

# 2. Load datasets.
df_train, X_train, y_train, ts_train = load_and_prepare_data(
    TRAIN_FILE,
    TARGET_COL_NAMES,
    COLS_TO_DROP_FROM_X
)

df_val, X_val, y_val, ts_val = load_and_prepare_data(
    VAL_FILE,
    TARGET_COL_NAMES,
    COLS_TO_DROP_FROM_X
)

df_test, X_test, y_test, ts_test = load_and_prepare_data(
    TEST_FILE,
    TARGET_COL_NAMES,
    COLS_TO_DROP_FROM_X
)

# Save column order for consistent feature order during plotting.
X_COLUMN_ORDER = X_train.columns

print("Data loading complete.")


## 11. Quick Optuna Test

This section runs a very small Optuna test.

Important notes:

- `sample_fraction = 0.1` means 10% of the training data is used.
- `initial_optuna_trials = 1` means only one trial is run.
- Because the objective function currently overwrites Optuna parameters with fixed parameters, this does not perform real tuning yet.

This part is useful for checking whether the pipeline runs, but it is not enough for final model tuning.


In [ ]:
print("\nStarting Optuna initial tuning...")

sample_fraction = 0.1

X_train_sampleXS = X_train.sample(frac=sample_fraction, random_state=42)
y_train_sampleXS = y_train.loc[X_train_sampleXS.index]

X_train_sample = X_train.sample(frac=sample_fraction, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

print(f"Using {len(X_train_sampleXS)} samples for initial Optuna tuning.")

initial_optuna_trials = 1

study_initial = optuna.create_study(
    direction="minimize",
    study_name="yssy_wind_initial_tuning"
)

study_initial.optimize(
    lambda trial: optuna_objective(
        trial,
        X_train_sampleXS,
        y_train_sampleXS,
        X_val,
        y_val
    ),
    n_trials=initial_optuna_trials
)

best_params_initial = study_initial.best_params

print(f"Best Initial Params after {initial_optuna_trials} trial(s): {best_params_initial}")
print(f"Best Initial Value: {study_initial.best_value}")

save_json(best_params_initial, BEST_PARAMS_INITIAL_FILE)


## 12. Train Model Using Saved Hyperparameters

This section loads hyperparameters from `output/best_params_final_31May.json` and trains a model.

In the current script, the model is called `temp_model`, but it is later saved as the final model.  
This naming is slightly confusing because the model is trained on only a sampled subset of the training data.

For a final training run, the model should ideally be trained on the full training dataset.


In [ ]:
best_hyperparameters = load_hyperparameters(PARAMS_FILE_PATH)

print("\nTraining temporary model for visualization and evaluation...")

temp_model = train_final_lgbm_model(
    X_train_sample,
    y_train_sample,
    best_hyperparameters
)

print("Model training complete.")


## 13. Manual Forecast Plot

This section attempts to plot a forecast for one manually specified timestamp.

The original script uses:

```python
df_test[TIMESTAMP_COL].iloc[0].strftime('2023-12-14 05:00')
```

This is unclear because `strftime()` usually expects a format string such as `"%Y-%m-%d %H:%M"`.

A clearer approach is to directly define the timestamp string:

```python
manual_plot_timestamp_str = "2023-12-14 05:00"
```


In [ ]:
if not df_test.empty:
    manual_plot_timestamp_str = "2023-12-14 05:00"

    try:
        manual_ts_to_plot = pd.to_datetime(manual_plot_timestamp_str)
        print(f"Attempting to plot specific forecast for timestamp: {manual_ts_to_plot}")

        target_row_df = df_test[df_test[TIMESTAMP_COL] == manual_ts_to_plot]

        if not target_row_df.empty:
            target_row = target_row_df.iloc[0]

            plot_single_forecast(
                manual_ts_to_plot,
                temp_model,
                target_row,
                TARGET_COL_NAMES,
                YSSY_U_LAG_COLS,
                YSSY_V_LAG_COLS,
                X_COLUMN_ORDER,
                plot_idx="_manual"
            )

        else:
            print(f"Timestamp {manual_ts_to_plot} not found in the test set.")

    except Exception as e:
        print(f"Error during manual plot for {manual_plot_timestamp_str}: {e}")

else:
    print("Test set is empty, skipping manual plot example.")


## 14. Random Forecast Plots

This section randomly selects samples from the test set and generates forecast plots.

Each plot compares:

- past input wind values
- actual future wind values
- model-predicted future wind values

The plots are saved into the `output/plots` folder.


In [ ]:
num_plot_samples = 25

plot_random_samples(
    temp_model,
    df_test,
    TARGET_COL_NAMES,
    YSSY_U_LAG_COLS,
    YSSY_V_LAG_COLS,
    X_COLUMN_ORDER,
    num_samples=num_plot_samples
)


## 15. Save Model and Evaluate Performance

This section saves the trained model and evaluates it on the test dataset.

The evaluation reports:

- overall MAE
- overall MSE
- per-target MAE and MSE

The current evaluation is based on `u` and `v` components.  
For a more weather-focused evaluation, future work could also convert `u/v` predictions into wind speed and wind direction.


In [ ]:
save_model_artifact(temp_model, FINAL_MODEL_FILE)

evaluate_model_performance(
    temp_model,
    X_test,
    y_test,
    TARGET_COL_NAMES
)

print("\nWorkflow complete.")
print(f"Outputs are in: {OUTPUT_DIR.resolve()}")


## 16. Reflection and Possible Improvements

Overall, this notebook shows a complete LightGBM-based workflow for YSSY wind forecasting. It loads prepared datasets, identifies future wind component targets, trains a multi-output model, evaluates model performance, and generates forecast plots.

However, the current workflow still looks more like an experimental version than a final training pipeline. The main issue is that the Optuna objective function defines a hyperparameter search space but then overwrites it with fixed parameters, so the tuning process does not actually explore different parameter combinations. In addition, only one Optuna trial is used, which is not enough for meaningful tuning.

Another issue is that the script saves the Optuna best parameters but trains the model using a separate parameter file. This makes the parameter workflow confusing. The model is also trained on only 10% of the training data, even though it is later saved as the final model. This is useful for quick testing, but for a final model, the full training dataset should be used.

There are also smaller issues, such as duplicated sampling code, unclear manual timestamp plotting logic, comments that do not match the actual code, and inconsistent naming between `temp_model` and the final saved model file.

For future improvement, the workflow should clearly separate quick testing from final training. The Optuna tuning logic should be fixed, the final model should be trained on the full dataset, and evaluation could be expanded from `u/v` component errors to wind speed and wind direction errors.
